# QAIFE Medical Image — PneumoniaMNIST

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vanbabenk/QAIFE-Medical-Image-Classification/blob/main/notebooks/QAIFE_Medical_Image_PneumoniaMNIST.ipynb)

This notebook implements the QAIFE configuration reported for **PneumoniaMNIST**. Set `RUN_MODE` to `single` for a quick independent run or `five` for the main five-run paper protocol. Both modes generate new random seeds automatically.


In [ ]:
# Dependencies and imports
import hashlib
import importlib.util
import random
import secrets
import subprocess
import sys
import time
import urllib.error
import urllib.parse
import urllib.request
from collections import deque
from copy import deepcopy
from functools import partial
from pathlib import Path

REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "torch": "torch",
    "torchvision": "torchvision",
    "pennylane": "pennylane==0.45.1",
    "medmnist": "medmnist==3.0.2",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
}

missing_packages = [
    pip_name
    for module_name, pip_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])

import matplotlib.pyplot as plt
import medmnist
import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
from IPython.display import display
from medmnist import INFO
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import ConcatDataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

print("Python    :", sys.version.split()[0])
print("PyTorch   :", torch.__version__)
print("PennyLane :", qml.__version__)
print("MedMNIST  :", medmnist.__version__)


In [ ]:
# QAIFE Medical Image configuration
PROTOCOL_NAME = "QAIFE Medical Image"
METHOD_NAME = "QAIFE"
DATASET_NAME = "PneumoniaMNIST"
DATASET_FLAG = "pneumoniamnist"
DATASET_CLASS_NAME = "PneumoniaMNIST"

DATA_ROOT = Path("./data/medmnist")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT = str(DATA_ROOT.resolve())
RESULT_DIR = Path("./outputs") / DATASET_NAME
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MEDMNIST_DOWNLOAD_TIMEOUT_SECONDS = 60
MEDMNIST_DOWNLOAD_RETRIES_PER_URL = 3
MEDMNIST_DOWNLOAD_BACKOFF_SECONDS = 1.0
MEDMNIST_DOWNLOAD_CHUNK_BYTES = 1024 * 1024
MEDMNIST_ARCHIVE_MD5 = "28209eda62fecd6e6a2d98b1501bb15f"
MEDMNIST_PINNED_DOWNLOAD_URLS = (
    "https://zenodo.org/records/10519652/files/{filename}?download=1",
    "https://zenodo.org/api/records/10519652/files/{filename}/content",
)
MEDMNIST_MIRROR_URL_TEMPLATES = (
    "https://huggingface.co/datasets/albertvillanova/medmnist-v2/"
    "resolve/main/data/{filename}?download=true",
)
MEDMNIST_EXTRA_DOWNLOAD_URLS = ()
MEDMNIST_SOURCE_SIZE = 28

EXPECTED_SPLIT_SIZES = {
    "train": 4708,
    "val": 524,
    "test": 624,
}

NUM_CLASSES = 2
IMAGE_SIZE = 22
PATCH_SIZE = 3
PATCH_STRIDE = 2
PATCHES_PER_SIDE = ((IMAGE_SIZE - PATCH_SIZE) // PATCH_STRIDE) + 1
N_PATCHES = PATCHES_PER_SIDE**2
PATCH_DIM = PATCH_SIZE * PATCH_SIZE
BATCH_SIZE = 64
LEARNING_RATE = 0.0005
MAX_ITERATIONS = 450

N_QUBITS_QFE = PATCH_DIM
N_LAYERS_QFE = 2
QFE_PROJECTED_DIM = 18
ATTENTION_INPUT_DIM = N_QUBITS_QFE
ATTENTION_DIM = ATTENTION_INPUT_DIM
N_QUBITS_ATTN = N_QUBITS_QFE + 1
ANCILLA_WIRE = N_QUBITS_QFE
CONCAT_DIM = QFE_PROJECTED_DIM + ATTENTION_DIM
CLASSIFIER_INPUT_DIM = CONCAT_DIM
FC_HIDDEN_DIMS = (128, 64)

QFE_INIT_LOW = 0.0
QFE_INIT_HIGH = 2.0 * np.pi
ATTENTION_INIT_LOW = 0.0
ATTENTION_INIT_HIGH = 2.0 * np.pi
ATTENTION_BETA = 1.0
FEATURE_RANGE_PHI = np.pi
FIXED_ATTENTION_PHASE = np.pi / 2.0

TRAIN_LOG_INTERVAL = max(1, min(25, MAX_ITERATIONS))
PROGRESS_WINDOW = min(25, MAX_ITERATIONS)
FC_BENCHMARK_BATCH_SIZE = 64
FC_BENCHMARK_WARMUP_STEPS = 50
FC_BENCHMARK_REPEATS = 1000

# Colab users can select "single" or "five" from this field.
RUN_MODE = "single"  # @param ["single", "five"]
AUTO_SEED_UPPER_BOUND = 2**31

def generate_auto_seeds(run_mode):
    if run_mode not in {"single", "five"}:
        raise ValueError("RUN_MODE must be either 'single' or 'five'.")
    run_count = 1 if run_mode == "single" else 5
    seeds = []
    while len(seeds) < run_count:
        candidate = secrets.randbelow(AUTO_SEED_UPPER_BOUND)
        if candidate not in seeds:
            seeds.append(candidate)
    return seeds

RUN_SEEDS = generate_auto_seeds(RUN_MODE)

QUANTUM_TORCH_DEVICE = torch.device("cpu")
CLASSIFIER_DEVICE = torch.device("cpu")
PIN_MEMORY = False

dataset_info = INFO[DATASET_FLAG]
if dataset_info["python_class"] != DATASET_CLASS_NAME:
    raise RuntimeError("MedMNIST dataset class does not match the configuration.")
if len(dataset_info["label"]) != NUM_CLASSES:
    raise RuntimeError("MedMNIST class count does not match the configuration.")

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Protocol       :", PROTOCOL_NAME)
print("Dataset        :", DATASET_NAME)
print("Run mode       :", RUN_MODE)
print("Generated runs :", len(RUN_SEEDS))
print("Update budget  :", MAX_ITERATIONS)
print("Preprocessing  : grayscale 22x22, normalized to [-1, 1]")


In [ ]:
# CELL 3 — MedMNIST splits, one-hot labels, and patch extraction
def calculate_file_md5(file_path):
    digest = hashlib.md5()
    with Path(file_path).open("rb") as file:
        while True:
            chunk = file.read(MEDMNIST_DOWNLOAD_CHUNK_BYTES)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def build_medmnist_download_urls():
    size_suffix = "" if MEDMNIST_SOURCE_SIZE == 28 else f"_{MEDMNIST_SOURCE_SIZE}"
    filename = f"{DATASET_FLAG}{size_suffix}.npz"
    urls = [
        template.format(
            filename=urllib.parse.quote(filename, safe=""),
            flag=DATASET_FLAG,
            size=MEDMNIST_SOURCE_SIZE,
            size_suffix=size_suffix,
        )
        for template in MEDMNIST_PINNED_DOWNLOAD_URLS
    ]

    if not urls:
        official_url = dataset_info[f"url{size_suffix}"]
        urls.append(official_url)

        parsed = urllib.parse.urlparse(official_url)
        path_parts = parsed.path.strip("/").split("/")
        if (
            len(path_parts) >= 4
            and path_parts[0] in {"record", "records"}
            and path_parts[2] == "files"
        ):
            record_id = path_parts[1]
            encoded_filename = urllib.parse.quote(
                urllib.parse.unquote("/".join(path_parts[3:])), safe=""
            )
            urls.append(
                f"{parsed.scheme}://{parsed.netloc}/api/records/{record_id}/"
                f"files/{encoded_filename}/content"
            )

    for template in MEDMNIST_MIRROR_URL_TEMPLATES:
        urls.append(
            template.format(
                filename=urllib.parse.quote(filename, safe=""),
                flag=DATASET_FLAG,
                size=MEDMNIST_SOURCE_SIZE,
                size_suffix=size_suffix,
            )
        )
    urls.extend(MEDMNIST_EXTRA_DOWNLOAD_URLS)
    return list(dict.fromkeys(url for url in urls if url))


def ensure_medmnist_archive():
    size_suffix = "" if MEDMNIST_SOURCE_SIZE == 28 else f"_{MEDMNIST_SOURCE_SIZE}"
    archive_name = f"{DATASET_FLAG}{size_suffix}.npz"
    archive_path = Path(DATA_ROOT) / archive_name
    expected_md5 = (
        MEDMNIST_ARCHIVE_MD5 or dataset_info[f"MD5{size_suffix}"]
    ).lower()

    if archive_path.is_file():
        cached_md5 = calculate_file_md5(archive_path)
        if cached_md5 == expected_md5:
            return archive_path
        print(
            f"Ignoring invalid cached archive {archive_path.name}: "
            f"expected MD5 {expected_md5}, received {cached_md5}."
        )

    partial_path = archive_path.with_suffix(archive_path.suffix + ".part")
    download_urls = build_medmnist_download_urls()
    last_error = None

    for source_index, url in enumerate(download_urls, start=1):
        for attempt in range(1, MEDMNIST_DOWNLOAD_RETRIES_PER_URL + 1):
            partial_path.unlink(missing_ok=True)
            try:
                request = urllib.request.Request(
                    url,
                    headers={
                        "User-Agent": "Mozilla/5.0 MedMNIST research notebook",
                        "Accept": "application/octet-stream",
                    },
                )
                with urllib.request.urlopen(
                    request,
                    timeout=MEDMNIST_DOWNLOAD_TIMEOUT_SECONDS,
                ) as response, partial_path.open("wb") as file:
                    while True:
                        chunk = response.read(MEDMNIST_DOWNLOAD_CHUNK_BYTES)
                        if not chunk:
                            break
                        file.write(chunk)

                received_md5 = calculate_file_md5(partial_path)
                if received_md5 != expected_md5:
                    raise RuntimeError(
                        "Downloaded archive failed MD5 verification: "
                        f"expected {expected_md5}, received {received_md5}."
                    )

                partial_path.replace(archive_path)
                print(
                    f"MedMNIST archive ready: {archive_path} "
                    f"(source {source_index}/{len(download_urls)})"
                )
                return archive_path
            except (
                urllib.error.HTTPError,
                urllib.error.URLError,
                TimeoutError,
                OSError,
                RuntimeError,
            ) as error:
                last_error = error
                partial_path.unlink(missing_ok=True)
                if attempt < MEDMNIST_DOWNLOAD_RETRIES_PER_URL:
                    delay_seconds = MEDMNIST_DOWNLOAD_BACKOFF_SECONDS * (
                        2 ** (attempt - 1)
                    )
                    print(
                        f"Download source {source_index}/{len(download_urls)} "
                        f"attempt {attempt} failed; retrying in "
                        f"{delay_seconds:g}s."
                    )
                    time.sleep(delay_seconds)

    partial_path.unlink(missing_ok=True)
    raise RuntimeError(
        f"Unable to download and verify {archive_name} after trying "
        f"{len(download_urls)} sources. Place the verified file in "
        f"{Path(DATA_ROOT)} and rerun this cell. Last error: {last_error}"
    ) from last_error


def extract_patches(images):
    """Convert (B, 1, 22, 22) images into (B, 100, 9) patches."""
    patches = images.unfold(2, PATCH_SIZE, PATCH_STRIDE).unfold(
        3, PATCH_SIZE, PATCH_STRIDE
    )
    return patches.contiguous().view(images.shape[0], N_PATCHES, PATCH_DIM)


def one_hot_target(target):
    target_array = np.asarray(target).reshape(-1)
    if target_array.size != 1:
        raise ValueError(
            f"Expected one class index, received shape {target_array.shape}."
        )

    class_index = int(target_array[0])
    if not 0 <= class_index < NUM_CLASSES:
        raise ValueError(f"Class index {class_index} is outside the valid range.")

    encoded = torch.zeros(NUM_CLASSES, dtype=torch.float32)
    encoded[class_index] = 1.0
    return encoded


def patch_collate_fn(batch):
    images, labels = zip(*batch)
    images = torch.stack(images, dim=0)
    labels = torch.stack(labels, dim=0).to(torch.float32)
    patches = extract_patches(images)
    return patches, labels


def build_loaders(seed):
    ensure_medmnist_archive()
    transform = transforms.Compose([
        transforms.Grayscale(num_output_channels=1),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,)),
    ])

    dataset_class = getattr(medmnist, DATASET_CLASS_NAME)
    dataset_kwargs = {
        "root": DATA_ROOT,
        "size": MEDMNIST_SOURCE_SIZE,
        "transform": transform,
        "target_transform": one_hot_target,
        "download": False,
        "as_rgb": False,
    }
    official_train = dataset_class(split="train", **dataset_kwargs)
    official_val = dataset_class(split="val", **dataset_kwargs)
    official_test = dataset_class(split="test", **dataset_kwargs)

    observed_split_sizes = {
        "train": len(official_train),
        "val": len(official_val),
        "test": len(official_test),
    }
    if observed_split_sizes != EXPECTED_SPLIT_SIZES:
        raise RuntimeError(
            "Official MedMNIST split sizes differ from the QAIFE Medical Image protocol: "
            f"expected {EXPECTED_SPLIT_SIZES}, received {observed_split_sizes}."
        )

    # Combine the official training and validation splits.
    train_pool = ConcatDataset([official_train, official_val])
    shuffle_generator = torch.Generator().manual_seed(seed)

    train_loader = DataLoader(
        train_pool,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=shuffle_generator,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        collate_fn=patch_collate_fn,
    )
    train_eval_loader = DataLoader(
        train_pool,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        collate_fn=patch_collate_fn,
    )
    test_loader = DataLoader(
        official_test,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        collate_fn=patch_collate_fn,
    )
    return train_loader, train_eval_loader, test_loader


In [ ]:
# QAIFE dual-operator, data-reuploading, ring-ZZ quantum extractor
def reupload_qaoa_ansatz(inputs, rz_params, rx_params):
    """Two-layer QAIFE extractor with RY+RX re-uploading in every layer."""
    rz_params = rz_params.reshape(N_LAYERS_QFE, N_QUBITS_QFE)
    rx_params = rx_params.reshape(N_LAYERS_QFE, N_QUBITS_QFE)

    for layer in range(N_LAYERS_QFE):
        # Dual-operator encoding and data re-uploading.
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS_QFE), rotation="Y")
        qml.AngleEmbedding(inputs, wires=range(N_QUBITS_QFE), rotation="X")

        for wire in range(N_QUBITS_QFE):
            qml.Hadamard(wires=wire)

        # QAOA-inspired ring entangler: CNOT-RZ-CNOT.
        for control in range(N_QUBITS_QFE):
            target = (control + 1) % N_QUBITS_QFE
            qml.CNOT(wires=[control, target])
            qml.RZ(rz_params[layer, control], wires=target)
            qml.CNOT(wires=[control, target])

        for wire in range(N_QUBITS_QFE):
            qml.RX(rx_params[layer, wire], wires=wire)


ZZ_RING_PAIRS = [
    (wire, (wire + 1) % N_QUBITS_QFE)
    for wire in range(N_QUBITS_QFE)
]

dev_qfe = qml.device("default.qubit", wires=N_QUBITS_QFE, shots=None)


@qml.qnode(dev_qfe, interface="torch", diff_method="backprop")
def qnode_qfe(inputs, weights_rz, weights_rx):
    reupload_qaoa_ansatz(inputs, weights_rz, weights_rx)
    return [
        qml.expval(qml.PauliZ(left_wire) @ qml.PauliZ(right_wire))
        for left_wire, right_wire in ZZ_RING_PAIRS
    ]


QFE_WEIGHT_SHAPES = {
    "weights_rz": (N_LAYERS_QFE, N_QUBITS_QFE),
    "weights_rx": (N_LAYERS_QFE, N_QUBITS_QFE),
}


In [ ]:
# Ancilla-based quantum attention
dev_attn = qml.device("default.qubit", wires=N_QUBITS_ATTN, shots=None)


@qml.qnode(dev_attn, interface="torch", diff_method="backprop")
def qnode_attn(inputs, attn_w):
    """
    Signed RY encoding -> H(ancilla) -> star-CRZ -> RZ(phi/2)
    -> H(ancilla) -> <Z_ancilla>.

    Each QAIFE ring-ZZ feature f is in [-1, 1] and is mapped monotonically
    to an angle in [0, pi]. The fixed ancilla phase is phi=pi, so the
    inserted phase-shift gate is RZ(pi/2).
    """
    for feature_wire in range(N_QUBITS_QFE):
        signed_angle = 0.5 * np.pi * (inputs[..., feature_wire] + 1.0)
        qml.RY(signed_angle, wires=feature_wire)

    qml.Hadamard(wires=ANCILLA_WIRE)
    for feature_wire in range(N_QUBITS_QFE):
        qml.CRZ(attn_w[feature_wire], wires=[feature_wire, ANCILLA_WIRE])
    qml.RZ(FIXED_ATTENTION_PHASE, wires=ANCILLA_WIRE)
    qml.Hadamard(wires=ANCILLA_WIRE)

    return qml.expval(qml.PauliZ(ANCILLA_WIRE))


ATTENTION_WEIGHT_SHAPES = {"attn_w": (ATTENTION_INPUT_DIM,)}


In [ ]:
# QAIFE feature fusion and classifier
def uniform_init(low, high):
    return partial(nn.init.uniform_, a=low, b=high)


def initialize_fc_layers(*layers):
    """Apply Kaiming initialization to the classifier."""
    for index, layer in enumerate(layers):
        nonlinearity = "relu" if index < len(layers) - 1 else "linear"
        nn.init.kaiming_normal_(layer.weight, mode="fan_in", nonlinearity=nonlinearity)
        nn.init.zeros_(layer.bias)


def quantum_activation(values):
    return torch.tanh(values) * torch.pi


class QAIFEExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        init_method = {
            "weights_rz": uniform_init(QFE_INIT_LOW, QFE_INIT_HIGH),
            "weights_rx": uniform_init(QFE_INIT_LOW, QFE_INIT_HIGH),
        }
        self.q_layer = qml.qnn.TorchLayer(
            qnode_qfe,
            QFE_WEIGHT_SHAPES,
            init_method=init_method,
        )
        self.project = nn.Linear(
            N_QUBITS_QFE * N_PATCHES,
            QFE_PROJECTED_DIM,
        )

    def forward(self, patches):
        batch_size, patch_count, patch_dim = patches.shape
        if (patch_count, patch_dim) != (N_PATCHES, PATCH_DIM):
            raise ValueError(
                f"Expected (B, {N_PATCHES}, {PATCH_DIM}), got {tuple(patches.shape)}."
            )

        flat = patches.reshape(batch_size * patch_count, patch_dim)
        patch_features = self.q_layer(flat).reshape(
            batch_size, patch_count, N_QUBITS_QFE
        )
        projected = quantum_activation(
            self.project(patch_features.reshape(batch_size, -1))
        )
        return projected, patch_features


class AncillaQuantumAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn_layer = qml.qnn.TorchLayer(
            qnode_attn,
            ATTENTION_WEIGHT_SHAPES,
            init_method={
                "attn_w": uniform_init(
                    ATTENTION_INIT_LOW,
                    ATTENTION_INIT_HIGH,
                )
            },
        )
    def forward(self, patch_features, return_details=False):
        batch_size, patch_count, feature_dim = patch_features.shape
        if (patch_count, feature_dim) != (N_PATCHES, ATTENTION_INPUT_DIM):
            raise ValueError(
                f"Expected (B, {N_PATCHES}, {ATTENTION_INPUT_DIM}), "
                f"got {tuple(patch_features.shape)}."
            )

        flat = patch_features.reshape(batch_size * patch_count, feature_dim)
        scores = self.attn_layer(flat).reshape(batch_size, patch_count)
        weights = torch.softmax(ATTENTION_BETA * scores, dim=1)

        # Softmax weights sum to one; use a weighted sum, never weighted mean.
        attention_native = torch.sum(
            patch_features * weights.unsqueeze(-1),
            dim=1,
        )
        # Fixed classical range matching: [-1,1] -> [-phi,phi].
        attention = attention_native * FEATURE_RANGE_PHI

        if return_details:
            return attention, attention_native, scores, weights
        return attention


class QAIFEClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.qfe = QAIFEExtractor().to(QUANTUM_TORCH_DEVICE)
        self.qattn = AncillaQuantumAttention().to(QUANTUM_TORCH_DEVICE)
        self.fc1 = nn.Linear(CLASSIFIER_INPUT_DIM, FC_HIDDEN_DIMS[0]).to(CLASSIFIER_DEVICE)
        self.fc2 = nn.Linear(FC_HIDDEN_DIMS[0], FC_HIDDEN_DIMS[1]).to(CLASSIFIER_DEVICE)
        self.fc3 = nn.Linear(FC_HIDDEN_DIMS[1], NUM_CLASSES).to(CLASSIFIER_DEVICE)
        initialize_fc_layers(self.fc1, self.fc2, self.fc3)

    def extract_features(self, patches, return_attention=False):
        patches = patches.to(QUANTUM_TORCH_DEVICE)
        projected, patch_features = self.qfe(patches)

        if return_attention:
            attention, attention_native, scores, weights = self.qattn(
                patch_features,
                return_details=True,
            )
        else:
            attention = self.qattn(patch_features)
            attention_native, scores, weights = None, None, None

        # Concatenate extractor and attention features.
        concatenated = torch.cat([projected, attention], dim=1)
        if concatenated.shape[1] != CONCAT_DIM:
            raise RuntimeError(
                f"Expected concatenated dimension {CONCAT_DIM}, "
                f"received {concatenated.shape[1]}."
            )

        if return_attention:
            return (
                concatenated, projected, attention, attention_native,
                scores, weights
            )
        return concatenated

    def classify(self, features):
        hidden = torch.relu(self.fc1(features))
        hidden = torch.relu(self.fc2(hidden))
        return self.fc3(hidden)

    def forward(self, patches):
        features = self.extract_features(patches)
        return self.classify(features.to(self.fc1.weight.device))


def count_parameters(module):
    return sum(parameter.numel() for parameter in module.parameters())


def parameter_breakdown(model):
    breakdown = {
        "QAIFE extractor": count_parameters(model.qfe),
        "Ancilla quantum attention": count_parameters(model.qattn.attn_layer),
        "FC1": count_parameters(model.fc1),
        "FC2": count_parameters(model.fc2),
        "FC3": count_parameters(model.fc3),
    }
    breakdown["Total"] = sum(breakdown.values())
    return breakdown


In [ ]:
# Evaluation, figures, and isolated FC Time
LOSS_FUNCTION = nn.CrossEntropyLoss()

def evaluate(model, data_loader):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    all_targets = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(QUANTUM_TORCH_DEVICE)
            targets = targets.to(CLASSIFIER_DEVICE)
            logits = model(inputs)
            loss = LOSS_FUNCTION(logits, targets)
            probabilities = torch.softmax(logits, dim=1)
            predictions = probabilities.argmax(dim=1)
            target_indices = targets.argmax(dim=1)

            batch_size = targets.shape[0]
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            all_targets.extend(target_indices.cpu().tolist())
            all_predictions.extend(predictions.cpu().tolist())
            all_probabilities.extend(probabilities.cpu().tolist())

    targets_array = np.asarray(all_targets, dtype=np.int64)
    predictions_array = np.asarray(all_predictions, dtype=np.int64)
    probabilities_array = np.asarray(all_probabilities, dtype=np.float64)
    expected_classes = np.arange(NUM_CLASSES)

    if NUM_CLASSES == 2:
        auc = roc_auc_score(targets_array, probabilities_array[:, 1])
    else:
        auc = roc_auc_score(
            targets_array,
            probabilities_array,
            labels=expected_classes,
            multi_class="ovr",
            average="macro",
        )

    metrics = {
        "loss": total_loss / total_samples,
        "accuracy": accuracy_score(targets_array, predictions_array),
        "precision": precision_score(
            targets_array, predictions_array, average="macro", zero_division=0
        ),
        "recall": recall_score(
            targets_array, predictions_array, average="macro", zero_division=0
        ),
        "f1": f1_score(
            targets_array, predictions_array, average="macro", zero_division=0
        ),
        "auc": auc,
    }
    return metrics, targets_array, predictions_array


class IsolatedFCClassifier(nn.Module):
    def __init__(self, source_model):
        super().__init__()
        self.fc1 = deepcopy(source_model.fc1)
        self.fc2 = deepcopy(source_model.fc2)
        self.fc3 = deepcopy(source_model.fc3)

    def forward(self, features):
        hidden = torch.relu(self.fc1(features))
        hidden = torch.relu(self.fc2(hidden))
        return self.fc3(hidden)


def synchronize_classifier():
    if CLASSIFIER_DEVICE.type == "cuda":
        torch.cuda.synchronize()


def measure_fc_time(model, classifier_input):
    if classifier_input.shape[0] != FC_BENCHMARK_BATCH_SIZE:
        raise ValueError(
            f"FC Time requires {FC_BENCHMARK_BATCH_SIZE} feature vectors."
        )

    fc_model = IsolatedFCClassifier(model).to(CLASSIFIER_DEVICE)
    fc_model.train()
    benchmark_input = classifier_input.detach().clone().to(CLASSIFIER_DEVICE)
    benchmark_targets = (
        torch.arange(FC_BENCHMARK_BATCH_SIZE, device=CLASSIFIER_DEVICE)
        % NUM_CLASSES
    )
    benchmark_optimizer = optim.Adam(fc_model.parameters(), lr=LEARNING_RATE)
    benchmark_loss = nn.CrossEntropyLoss()

    def update_fc():
        benchmark_optimizer.zero_grad(set_to_none=True)
        logits = fc_model(benchmark_input)
        loss = benchmark_loss(logits, benchmark_targets)
        loss.backward()
        benchmark_optimizer.step()

    for _ in range(FC_BENCHMARK_WARMUP_STEPS):
        update_fc()

    synchronize_classifier()
    start_time = time.perf_counter()
    for _ in range(FC_BENCHMARK_REPEATS):
        update_fc()
    synchronize_classifier()
    return time.perf_counter() - start_time


def save_figure(fig, filename):
    fig.savefig(RESULT_DIR / filename, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()


def plot_training(history):
    fig, loss_axis = plt.subplots(figsize=(7.0, 4.5), constrained_layout=True)
    accuracy_axis = loss_axis.twinx()
    loss_line = loss_axis.plot(
        history["iteration"], history["batch_loss"],
        color="#D97706", linewidth=1.4, label="Train Loss"
    )
    accuracy_line = accuracy_axis.plot(
        history["iteration"], history["batch_accuracy"],
        color="#15803D", linewidth=1.4, label="Train Accuracy"
    )
    loss_axis.set_xlabel("Iteration")
    loss_axis.set_ylabel("Loss", color="#D97706")
    accuracy_axis.set_ylabel("Accuracy", color="#15803D")
    loss_axis.set_ylim(0.0, 3.0)
    accuracy_axis.set_ylim(0.0, 1.0)
    lines = loss_line + accuracy_line
    loss_axis.legend(lines, [line.get_label() for line in lines], loc="center right")
    loss_axis.set_title(f"QAIFE Training on {DATASET_NAME}")
    save_figure(fig, f"QAIFE_{DATASET_NAME}_Training.png")


def plot_confusion(y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    display_matrix = ConfusionMatrixDisplay(
        matrix, display_labels=list(range(NUM_CLASSES))
    )
    fig, ax = plt.subplots(figsize=(6.2, 5.4), constrained_layout=True)
    display_matrix.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"QAIFE - {DATASET_NAME}")
    save_figure(fig, f"QAIFE_{DATASET_NAME}_Confusion_Matrix.png")


In [ ]:
# Training and experiment execution
def train_one_run(seed):
    set_seed(seed)
    train_loader, train_eval_loader, test_loader = build_loaders(seed)
    model = QAIFEClassifier()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    history = {"iteration": [], "batch_loss": [], "batch_accuracy": []}
    rolling_losses = deque(maxlen=PROGRESS_WINDOW)
    rolling_accuracies = deque(maxlen=PROGRESS_WINDOW)
    train_iterator = iter(train_loader)
    fc_input = None

    progress = tqdm(range(1, MAX_ITERATIONS + 1), desc=f"QAIFE {DATASET_NAME}")
    for iteration in progress:
        try:
            patches, targets = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            patches, targets = next(train_iterator)

        patches = patches.to(QUANTUM_TORCH_DEVICE)
        targets = targets.to(CLASSIFIER_DEVICE)
        target_indices = targets.argmax(dim=1)

        model.train()
        optimizer.zero_grad(set_to_none=True)
        features = model.extract_features(patches)
        classifier_input = features.to(CLASSIFIER_DEVICE).contiguous()
        if fc_input is None:
            fc_input = classifier_input[:FC_BENCHMARK_BATCH_SIZE].detach().clone()
        logits = model.classify(classifier_input)
        loss = LOSS_FUNCTION(logits, targets)
        loss.backward()
        optimizer.step()

        batch_accuracy = (logits.argmax(dim=1) == target_indices).float().mean().item()
        history["iteration"].append(iteration)
        history["batch_loss"].append(float(loss.item()))
        history["batch_accuracy"].append(float(batch_accuracy))
        rolling_losses.append(float(loss.item()))
        rolling_accuracies.append(float(batch_accuracy))

        if iteration == 1 or iteration % TRAIN_LOG_INTERVAL == 0 or iteration == MAX_ITERATIONS:
            progress.set_postfix(
                loss=f"{np.mean(rolling_losses):.4f}",
                accuracy=f"{np.mean(rolling_accuracies):.4f}",
            )

    if fc_input is None or fc_input.shape[0] != FC_BENCHMARK_BATCH_SIZE:
        raise RuntimeError("Unable to capture a complete FC Time input batch.")

    train_metrics, _, _ = evaluate(model, train_eval_loader)
    test_metrics, y_true, y_pred = evaluate(model, test_loader)
    fc_time = measure_fc_time(model, fc_input)

    result = {
        "seed": seed,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_precision": train_metrics["precision"],
        "train_recall": train_metrics["recall"],
        "train_f1": train_metrics["f1"],
        "train_auc": train_metrics["auc"],
        "test_loss": test_metrics["loss"],
        "test_accuracy": test_metrics["accuracy"],
        "test_precision": test_metrics["precision"],
        "test_recall": test_metrics["recall"],
        "test_f1": test_metrics["f1"],
        "test_auc": test_metrics["auc"],
        "fc_time_seconds": fc_time,
    }
    return result, history, y_true, y_pred


all_results = []
run_records = []
print("Automatically generated run seeds:", RUN_SEEDS)

for run_index, seed in enumerate(RUN_SEEDS, start=1):
    result, history, y_true, y_pred = train_one_run(seed)
    all_results.append(result)
    run_records.append({
        "result": result,
        "history": history,
        "y_true": y_true,
        "y_pred": y_pred,
    })
    print(f"\nRun {run_index}/{len(RUN_SEEDS)} | seed={seed}")
    print(
        f"Train | loss={result['train_loss']:.4f} | "
        f"accuracy={result['train_accuracy']:.4f} | "
        f"precision={result['train_precision']:.4f} | "
        f"recall={result['train_recall']:.4f} | "
        f"F1={result['train_f1']:.4f} | AUC={result['train_auc']:.4f}"
    )
    print(
        f"Test | loss={result['test_loss']:.4f} | "
        f"accuracy={result['test_accuracy']:.4f} | "
        f"precision={result['test_precision']:.4f} | "
        f"recall={result['test_recall']:.4f} | "
        f"F1={result['test_f1']:.4f} | AUC={result['test_auc']:.4f}"
    )
    print(f"FC Time: {result['fc_time_seconds']:.2f} s")

results_df = pd.DataFrame(all_results)
summary_metrics = [
    "train_loss",
    "train_accuracy",
    "train_precision",
    "train_recall",
    "train_f1",
    "train_auc",
    "test_loss",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_auc",
    "fc_time_seconds",
]

summary = []
for metric in summary_metrics:
    values = results_df[metric].astype(float)
    summary.append({
        "Metric": "FC Time (s)" if metric == "fc_time_seconds" else metric,
        "Mean": values.mean(),
        "Sample SD": values.std(ddof=1) if len(values) > 1 else np.nan,
    })
display(pd.DataFrame(summary))

if len(results_df) > 1:
    print(
        f"FC Time (s): {results_df['fc_time_seconds'].mean():.2f} ± "
        f"{results_df['fc_time_seconds'].std(ddof=1):.2f}"
    )
else:
    print(f"FC Time (s): {results_df['fc_time_seconds'].iloc[0]:.2f}")

best_visual_run = max(
    run_records,
    key=lambda record: record["result"]["test_accuracy"],
)
plot_training(best_visual_run["history"])
plot_confusion(best_visual_run["y_true"], best_visual_run["y_pred"])
print(
    "Figures use the run with the highest test accuracy; "
    "numeric summaries use all completed runs."
)
